<div align="center">
  <img src="https://raw.githubusercontent.com/harishs-io/lateral_flowimmunoassay_quant/main/assets/lfia_lod_banner.png" width="950">
  <h1>🧬 LFIA Signal Quantification & LoD Calculation</h1>
</div>

---
**Instructions:**
1. Run the **Environment Setup** cell to load the pipeline.
2. Run the **Upload & Analyze** cell to insert your image and extract the signal.
3. Run the **Export** cell to download your data.

In [ ]:
#@title ⚙️ 1. Environment Setup { display-mode: "form" }
#@markdown Run this cell once to configure the workspace. Installation logs will be displayed below.

!git clone https://github.com/harishs-io/lateral_flowimmunoassay_quant.git
%cd lateral_flowimmunoassay_quant
!pip install ultralytics opencv-python matplotlib scipy pandas numpy

import os
import sys
sys.path.append('.')
from google.colab import files
from IPython.display import Image, display

print("\n✅ Pipeline environment successfully loaded! Move to Step 2.")

In [ ]:
#@title 🔬 2. Upload Image & Run Pipeline { display-mode: "form" }
#@markdown Adjust the detection parameters and choose whether to fine-tune the model with your images.

yolo_confidence = 0.20 #@param {type:"slider", min:0.1, max:1.0, step:0.05}
fine_tune_yolo = "No" #@param ["Yes", "No"]

import shutil
import builtins
import glob
import os
import io
import re
from contextlib import redirect_stdout
from IPython.display import clear_output, Image, display, HTML

# Map the UI dropdown to the script's input prompt
auto_response = 'y' if fine_tune_yolo == "Yes" else 'n'
builtins.input = lambda prompt: auto_response

print("👇 Click 'Choose Files' below to insert your assay image(s):")
uploaded = files.upload()

if uploaded:
    clear_output()
    print(f"✅ Successfully uploaded {len(uploaded)} image(s)!")
    print("⏳ Processing images through YOLOv8 and extracting chromatograms... (This may take a moment)\n")
    
    # Recreate the folder structure the script expects
    base_dir = "colab_run"
    sub_dir = os.path.join(base_dir, "assay_01")
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
    os.makedirs(sub_dir, exist_ok=True)
    
    # Move ALL uploaded images to the target directory
    for image_name in uploaded.keys():
        shutil.move(image_name, os.path.join(sub_dir, image_name))
    
    try:
        from src.lfia_analyzer import run_pipeline
        
        # Trap the messy terminal logs so they don't print to the UI
        trap = io.StringIO()
        with redirect_stdout(trap):
            run_pipeline(base_dir=base_dir, dataset_dir="dataset", weights_path="weights/best.pt", conf_thresh=yolo_confidence)
        
        print("✅ Pipeline complete! Targets localized and integrated.\n")
        print("---")
        
        # Find, parse, and display clean results
        txt_files = glob.glob(f"{sub_dir}/Results/**/*_results.txt", recursive=True)
        txt_files.sort() # Display in order
        
        for txt_file in txt_files:
            with open(txt_file, 'r') as f:
                content = f.read()
                
            # Extract values using regex
            conc_match = re.search(r'Concentration:\s*(.*)', content)
            t_match = re.search(r'Test \(T\) Area:\s*([0-9.]+)', content)
            c_match = re.search(r'Control \(C\) Area:\s*([0-9.]+)', content)
            tc_match = re.search(r'T/C Ratio:\s*([0-9.]+)', content)
            
            conc = conc_match.group(1).strip() if conc_match else "Unknown"
            t_val = t_match.group(1) if t_match else "N/A"
            c_val = c_match.group(1) if c_match else "N/A"
            tc_val = tc_match.group(1) if tc_match else "N/A"
            
            # Print clean formatted UI row
            print(f"🔬 Analyzed: {conc} ng/mL | T value: {t_val} | C value: {c_val} | T/C: {tc_val}")
            
            # Display scaled-down image directly below the stat row
            plot_file = txt_file.replace('_results.txt', '_Analysis_Plot.png')
            if os.path.exists(plot_file):
                display(Image(filename=plot_file, width=650))
                print("") # Add spacing
                
        # Check if a Standard Curve was generated (requires multiple points)
        std_curve = glob.glob(f"{sub_dir}/Results/*_Standard_Curve_Plot.png")
        if std_curve:
            print("\n📈 Standard Curve Generated:")
            display(Image(filename=std_curve[0], width=500))
            
    except Exception as e:
        print(f"\n⚠️ Pipeline Error: {e}")
else:
    print("⚠️ No file uploaded. Please run the cell again.")

In [ ]:
#@title 📊 3. Export Data & Reports { display-mode: "form" }
#@markdown Zips the generated `Results/` folder (including cropped images, text reports, and plots) and downloads it to your machine.

try:
    shutil.make_archive("LFIA_Results", 'zip', "colab_run/assay_01/Results")
    print("✅ Results packaged. Triggering download for 'LFIA_Results.zip'...")
    files.download('LFIA_Results.zip')
except Exception as e:
    print("⚠️ Error: Please run Step 2 first to generate the results.")